In [3]:
import sys
print(sys.executable)

import torch
print(torch.__version__)

/home/sduai/.virtualenvs/nids-research/bin/python
2.11.0+cu128


In [8]:
import sys
from pathlib import Path

ROOT = Path("/mnt/c/users/sduai/documents/projis/research/nids_dl/nids-research")
sys.path.insert(0, str(ROOT / "src"))

import torch


from nids_dl import RFConfig, TrainConfig, evaluate, extract_features, fit_rf, train_extractor
from nids_dl.data import load_processed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
  print(f"Using {torch.cuda.device_count()} GPU(s):")
  for i in range(torch.cuda.device_count()):
      print(f"  cuda:{i}  {torch.cuda.get_device_name(i)}")
else:
  print("No GPU found, using CPU")


Using 2 GPU(s):
  cuda:0  NVIDIA GeForce RTX 5090
  cuda:1  NVIDIA GeForce RTX 5090


In [9]:
DEVICE

'cuda'

In [10]:
ROOT

PosixPath('/mnt/c/users/sduai/documents/projis/research/nids_dl/nids-research')

## 1. Load preprocessed NSL-KDD

In [11]:
train = load_processed(ROOT / "data" / "processed" / "train.pt")
test = load_processed(ROOT / "data" / "processed" / "test.pt")

X_tr, y_tr = train["X"], train["y_bin"]
X_te, y_te = test["X"], test["y_bin"]
X_tr.shape, X_te.shape, int(y_tr.max().item()) + 1

(torch.Size([125973, 120]), torch.Size([22544, 120]), 2)

## 2. Phase 1 — train the DL feature extractor

Cross-entropy on the temporary softmax head, Adam, with the MCL prediction-error-filter constraint re-applied after every step.

In [14]:
cfg = TrainConfig(epochs=10, batch_size=256, lr=1e-3, device=DEVICE, target="binary", log_every=0, seed=0)
extractor, history = train_extractor(X_tr, y_tr, cfg, X_val=X_te, y_val=y_te)
history[-1]

{'epoch': 9,
 'train_loss': 0.06854488520208697,
 'train_acc': 0.9764235193255698,
 'val_loss': 0.8314211868579884,
 'val_acc': 0.8118346344925479}

In [13]:
import pandas as pd

pd.DataFrame(history)

,epoch,train_loss,train_acc,val_loss,val_acc
0,0,0.261019,0.888198,0.567232,0.790676
1,1,0.175653,0.946822,2.149576,0.430758
2,2,0.258731,0.901487,0.905834,0.663281
3,3,0.188354,0.933319,0.491633,0.813210
4,4,0.156694,0.949037,0.854907,0.723518
5,5,0.170475,0.942289,0.470397,0.800790
6,6,0.156086,0.947282,0.619857,0.756831
7,7,0.161030,0.946830,0.880772,0.715623
8,8,0.149140,0.952442,0.722905,0.751242
9,9,0.134780,0.957920,0.690973,0.734120


## 3. Extract features and fit RandomForest

In [15]:
Fe_tr = extract_features(extractor, X_tr, batch_size=512, device=DEVICE).numpy()
Fe_te = extract_features(extractor, X_te, batch_size=512, device=DEVICE).numpy()
Fe_tr.shape, Fe_te.shape

((125973, 256), (22544, 256))

In [16]:
clf = fit_rf(Fe_tr, y_tr.numpy(), RFConfig(n_estimators=200, class_weight="balanced", random_state=0))
metrics_train = evaluate(clf, Fe_tr, y_tr.numpy())
metrics_test = evaluate(clf, Fe_te, y_te.numpy())
{
    "train": {k: metrics_train[k] for k in ("accuracy", "precision", "recall", "f1")},
    "test": {k: metrics_test[k] for k in ("accuracy", "precision", "recall", "f1")},
}

{'train': {'accuracy': 0.999944432537131,
  'precision': 0.9999488307834007,
  'recall': 0.9999317755415317,
  'f1': 0.9999403030897415},
 'test': {'accuracy': 0.7824698367636622,
  'precision': 0.9680084995868257,
  'recall': 0.6389776357827476,
  'f1': 0.7698084866691701}}

In [17]:
print(metrics_test["report"])
metrics_test["confusion_matrix"]

              precision    recall  f1-score   support

           0       0.67      0.97      0.79      9711
           1       0.97      0.64      0.77     12833

    accuracy                           0.78     22544
   macro avg       0.82      0.81      0.78     22544
weighted avg       0.84      0.78      0.78     22544



array([[9440,  271],
       [4633, 8200]])

## 4. Multi-class variant (5 classes: Normal/DoS/Probe/R2L/U2R)

In [ ]:
ym_tr, ym_te = train["y_mul"], test["y_mul"]
cfg_m = TrainConfig(epochs=10, batch_size=256, lr=1e-3, device=DEVICE, target="multi", seed=0)
extractor_m, history_m = train_extractor(X_tr, ym_tr, cfg_m, X_val=X_te, y_val=ym_te)
Fe_tr_m = extract_features(extractor_m, X_tr, batch_size=512, device=DEVICE).numpy()
Fe_te_m = extract_features(extractor_m, X_te, batch_size=512, device=DEVICE).numpy()
clf_m = fit_rf(Fe_tr_m, ym_tr.numpy(), RFConfig(n_estimators=200, class_weight="balanced", random_state=0))
metrics_te_m = evaluate(clf_m, Fe_te_m, ym_te.numpy())
print(metrics_te_m["report"])
{k: metrics_te_m[k] for k in ("accuracy", "precision", "recall", "f1")}